In [28]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300
plt.rcParams['figure.facecolor'] = 'darkgrey'

In [29]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.sel(time=slice('1992-01-01', '2024-12-31')).to_dataframe().reset_index()
sst_df = sst_df.query('time_bnds > 0 and nbnds == 0')
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year

In [30]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [31]:
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst['std'] = sst_df.dropna().groupby(['lat', 'lon', 'month']).std().reset_index()['sst']
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [32]:
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']
sst_anomaly['normalized_sst_anomaly'] = sst_anomaly['sst_anomaly'] / sst_anomaly['std']

In [33]:
chirps_eastern_east_africa = chirps.sel(time=slice('1993-01-01', '2024-01-01'), latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

month_to_season = {
    3: 'MAM', 4: 'MAM', 5: 'MAM',   # March, April, May
}

season = ['MAM']

chirps_eastern_east_africa['season'] = chirps_eastern_east_africa['month'].map(month_to_season)

chirps_eastern_east_africa = chirps_eastern_east_africa.dropna(subset=['season'])

chirps_eastern_east_africa_monthly = chirps_eastern_east_africa.groupby(['year', 'month'])['precip'].mean().reset_index()

chirps_eastern_east_africa_season = chirps_eastern_east_africa.groupby(['year', 'season'])[['precip']].mean().reset_index()

In [34]:
def get_tercile_labels(chirps):
    tercile_list = chirps.quantile([0.33, 0.66], numeric_only=True)['precip'].to_list()
    bn_list = chirps.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    n_list = chirps.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    an_list = chirps.query(f'{tercile_list[1]} <= precip')['year'].to_list()
    season_dict = {'an': an_list, 'bn': bn_list, 'n': n_list}

    year_to_category_map = {}
    for category, year_list in season_dict.items():
        for year in year_list:
            year_to_category_map[year] = category

    chirps['tercile'] = chirps['year'].map(year_to_category_map)

    return chirps

In [64]:
labeled_chirps_seasonal = get_tercile_labels(chirps_eastern_east_africa_season)
labeled_chirps_monthly = chirps_eastern_east_africa_monthly.merge(labeled_chirps_seasonal[['year', 'tercile']], how='left', on='year')

In [93]:
nino_34 = sst_anomaly.query('-5 <= lat <= 5 and -170<= lon <= -120').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().rename({'normalized_sst_anomaly': 'nino_34'}, axis=1)

nino_4 = sst_anomaly.query('-5 <= lat <= 5 and lon <= -150 or -5 <= lat <= 5 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'nino_4'}, axis=1)

western_west_v = sst_anomaly.query('-15 <= lat <= 20 and 120 <= lon <= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'western_west_v'}, axis=1)

northern_west_v = sst_anomaly.query('20 <= lat <= 35 and lon <= -150 or 20 <= lat <= 35 and lon >= 160').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'northern_west_v'}, axis=1)

southern_west_v = sst_anomaly.query('-30 <= lat <= -15 and lon <= -150 or -30 <= lat <= -15 and lon >= 155').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'southern_west_v'}, axis=1)

SWIO = sst_anomaly.query('-50 <= lat <= -20 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'SWIO'}, axis=1)

IOD_west = sst_anomaly.query('-10 <= lat <= 10 and 50 <= lon <= 70').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_west'}, axis=1)

IOD_east = sst_anomaly.query('-10 <= lat <= 0 and 90 <= lon <= 110').groupby(['year', 'month'])['normalized_sst_anomaly'].mean().reset_index().drop(['month', 'year'], axis=1).rename({'normalized_sst_anomaly': 'IOD_east'}, axis=1)

predictors = pd.concat([nino_34, nino_4, western_west_v, northern_west_v, southern_west_v, SWIO, IOD_west, IOD_east], axis=1)

In [92]:
five_month_lead = {10: 'MAM'}
eight_month_lead = {7: 'MAM'}

predictors_5_lead = predictors.copy()
predictors_8_lead = predictors.copy()
predictors_5_lead['effect_season'] = predictors_5_lead['month'].map(five_month_lead)
predictors_8_lead['effect_season'] = predictors_8_lead['month'].map(eight_month_lead)

predictors_with_lead = predictors_8_lead.dropna(subset=['effect_season']).drop('month', axis=1).merge(predictors_5_lead.dropna(subset=['effect_season']).drop('month', axis=1), on=['year', 'effect_season'], suffixes=('_8_lead', '_5_lead'))

predictors_with_lead['effect_year'] = predictors_with_lead['year'] + 1

ml_data_seasonal = labeled_chirps_seasonal.merge(predictors_with_lead.drop(['year'], axis=1), left_on=['year', 'season'], right_on=['effect_year', 'effect_season'], how='left').drop(['effect_year', 'effect_season'], axis=1)